In [1]:
import os
os.environ["HF_HOME"] = "/home/ltw/files/GraduationThesis/models"
os.environ["HF_HUB_CACHE"] = "/home/ltw/files/GraduationThesis/models/hub"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 检查 GPU 是否可用
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
if torch.cuda.is_available():
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
else:
    print("警告: 未检测到 GPU，将使用 CPU 训练（速度较慢）")

使用设备: cuda
GPU 名称: NVIDIA GeForce RTX 3050 Laptop GPU
GPU 数量: 1


In [2]:
# 安装必要的依赖库（使用清华大学镜像源）
# 安装必要的依赖库（使用清华大学镜像源）
# 安装必要的依赖库（使用清华大学镜像源）


### 数据集与模型
- 这些数据集都可以直接用：https://huggingface.co/datasets 
- 咱们今天玩这个(GLUE):https://gluebenchmark.com/

In [3]:
import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset#https://github.com/huggingface/datasets
import time

# 加载数据集，添加重试机制和延迟以避免 429 错误
max_retries = 3
retry_delay = 5  # 秒

for attempt in range(max_retries):
    try:
        # 使用 reuse_cache_if_exists 模式，如果已有缓存则直接使用
        raw_datasets = load_dataset("glue", "mrpc", download_mode="reuse_cache_if_exists")
        print("数据集加载成功！")
        break
    except Exception as e:
        if "429" in str(e) or "Too Many Requests" in str(e):
            if attempt < max_retries - 1:
                print(f"遇到请求限制，等待 {retry_delay} 秒后重试... (尝试 {attempt + 1}/{max_retries})")
                time.sleep(retry_delay)
                retry_delay *= 2  # 指数退避
            else:
                print("达到最大重试次数，请稍后再试或使用已缓存的数据")
                raise
        else:
            raise

raw_datasets

Using the latest cached version of the dataset since glue couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'mrpc' at /home/ltw/files/GraduationThesis/models/datasets/glue/mrpc/0.0.0/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c (last modified on Wed Jan  7 19:59:53 2026).


数据集加载成功！


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

看看数据长啥样子

In [4]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[100]

{'sentence1': 'The Nasdaq composite index inched up 1.28 , or 0.1 percent , to 1,766.60 , following a weekly win of 3.7 percent .',
 'sentence2': 'The technology-laced Nasdaq Composite Index .IXIC was off 24.44 points , or 1.39 percent , at 1,739.87 .',
 'label': 0,
 'idx': 114}

In [5]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

### 使用AutoTokenizer来处理数据

In [6]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

不是所有模型返回结果都一样的，得看你选择模型训练的时候人家咋设置的

In [7]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

{'input_ids': [101, 2023, 2003, 1996, 2034, 6251, 1012, 102, 2023, 2003, 1996, 2117, 2028, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

['[CLS]',
 'this',
 'is',
 'the',
 'first',
 'sentence',
 '.',
 '[SEP]',
 'this',
 'is',
 'the',
 'second',
 'one',
 '.',
 '[SEP]']

### 对所有数据进行处理


In [9]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [10]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 408/408 [00:00<00:00, 13776.84 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [11]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [12]:
tokenized_datasets["train"][0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0,
 'input_ids': [101,
  2572,
  3217,
  5831,
  5496,
  2010,
  2567,
  1010,
  3183,
  2002,
  2170,
  1000,
  1996,
  7409,
  1000,
  1010,
  1997,
  9969,
  4487,
  23809,
  3436,
  2010,
  3350,
  1012,
  102,
  7727,
  2000,
  2032,
  2004,
  2069,
  1000,
  1996,
  7409,
  1000,
  1010,
  2572,
  3217,
  5831,
  5496,
  2010,
  2567,
  1997,
  9969,
  4487,
  23809,
  3436,
  2010,
  3350,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
 

In [13]:
samples = tokenized_datasets["train"][:8]#取到所有的列
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}#不需要这些列
[len(x) for x in samples["input_ids"]]#每一个样本的长度

[50, 59, 47, 67, 59, 50, 62, 32]

经过data_collator处理之后，所有的样本长度都是固定的

In [14]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

### 训练模块

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

API文档：实际用的时候一定对应着来
- https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments

In [16]:
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=no,
eval_use_gather_object=False,
fp16=False,
fp16_b

In [17]:
from transformers import AutoModelForSequenceClassification

# 加载模型并移动到 GPU
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model = model.to(device)  # 将模型移动到 GPU
print(f"模型已加载到: {next(model.parameters()).device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


模型已加载到: cuda:0


In [18]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

### 训练与评估的区别（重要：工程完整性）

**关键点：**
- `trainer.train()` 仅输出训练集损失（看模型"学没学会"）
- 验证集的准确率 / F1 值属于评估指标，需要手动配置"指标计算规则" + 调用 `trainer.evaluate()` 才能输出
- 这是复试老师会关注的"工程完整性"细节（证明你懂"训练"和"评估"的区别）

**为什么重要：**
- 训练损失只告诉我们模型在训练数据上的表现
- 评估指标（准确率、F1等）才能反映模型的真实泛化能力
- 完整的训练流程应该包括：训练 → 评估 → 分析结果


In [20]:
# 训练前验证 GPU 使用情况
print(f"模型设备: {next(trainer.model.parameters()).device}")
print(f"开始训练...")

# 训练模型
trainer.train()

# 重要：训练后进行评估（工程完整性）
# trainer.train() 只输出训练损失，要查看评估指标需要调用 trainer.evaluate()
print("\n=== 开始评估 ===")
eval_results = trainer.evaluate()
print(f"评估结果: {eval_results}")

模型设备: cuda:0
开始训练...


Step,Training Loss
500,0.443400
1000,0.327300



=== 开始评估 ===


评估结果: {'eval_loss': 0.7436676621437073, 'eval_runtime': 2.0783, 'eval_samples_per_second': 196.311, 'eval_steps_per_second': 24.539, 'epoch': 3.0}


In [21]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


In [22]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

In [23]:
from evaluate import load

metric = load("glue", config_name="mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8235294117647058, 'f1': 0.8771331058020477}

### 训练过程中也可以指定好评估方法

In [ ]:
from evaluate import load

def compute_metrics(eval_preds):
    metric = load("glue", config_name="mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments("test-trainer")
# 加载模型并移动到 GPU
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model = model.to(device)  # 将模型移动到 GPU
print(f"模型已加载到: {next(model.parameters()).device}")

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


模型已加载到: cuda:0


会把每一个epoch的评估结果进行返回

In [ ]:
# 训练前验证 GPU 使用情况
print(f"模型设备: {next(trainer.model.parameters()).device}")
print(f"开始训练...")

# 训练模型
trainer.train()

# 重要：训练后进行评估（工程完整性）
# trainer.train() 只输出训练损失，要查看评估指标需要调用 trainer.evaluate()
print("\n=== 开始评估 ===")
eval_results = trainer.evaluate()
print(f"评估结果: {eval_results}")

模型设备: cuda:0
开始训练...


Step,Training Loss
500,0.587500
1000,0.428400



=== 开始评估 ===


评估结果: {'eval_loss': 0.5513553023338318, 'eval_accuracy': 0.8455882352941176, 'eval_f1': 0.8941176470588236, 'eval_runtime': 4.2377, 'eval_samples_per_second': 96.278, 'eval_steps_per_second': 12.035, 'epoch': 3.0}


### 补充实验：不使用 DataCollatorWithPadding，自己写预处理

对比使用 `DataCollatorWithPadding` 和自定义预处理函数的效果差异

In [25]:
import torch
import numpy as np

def custom_collate_fn(batch):
    """
    自定义的批处理函数，手动实现 padding
    不使用 DataCollatorWithPadding，看看效果差多少
    
    注意：这个函数手动实现了 DataCollatorWithPadding 的功能
    用于对比实验，理解底层实现原理
    """
    # 获取 tokenizer 的 pad_token_id（用于 padding）
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    
    # 获取所有样本的 input_ids（转换为列表格式）
    input_ids_list = []
    attention_mask_list = []
    token_type_ids_list = []
    labels_list = []
    
    for item in batch:
        # 处理 input_ids（可能是 list 或 tensor）
        ids = item["input_ids"]
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        input_ids_list.append(ids)
        
        # 处理 attention_mask
        mask = item.get("attention_mask", [1] * len(ids))
        if isinstance(mask, torch.Tensor):
            mask = mask.tolist()
        attention_mask_list.append(mask)
        
        # 处理 token_type_ids（如果有）
        if "token_type_ids" in item:
            token_type_ids = item["token_type_ids"]
            if isinstance(token_type_ids, torch.Tensor):
                token_type_ids = token_type_ids.tolist()
            token_type_ids_list.append(token_type_ids)
        
        # 处理 labels（如果有）
        if "label" in item:
            labels_list.append(item["label"])
    
    # 找到批次中最长的序列长度
    max_length = max(len(ids) for ids in input_ids_list)
    
    # 手动进行 padding
    padded_input_ids = []
    padded_attention_mask = []
    padded_token_type_ids = []
    
    for i, (ids, mask) in enumerate(zip(input_ids_list, attention_mask_list)):
        # 计算需要 padding 的长度
        pad_length = max_length - len(ids)
        
        # 对 input_ids 进行 padding
        padded_ids = ids + [pad_token_id] * pad_length
        padded_input_ids.append(padded_ids)
        
        # 对 attention_mask 进行 padding（0 表示 padding 位置）
        padded_mask = mask + [0] * pad_length
        padded_attention_mask.append(padded_mask)
        
        # 对 token_type_ids 进行 padding（如果有）
        if token_type_ids_list:
            token_type_ids = token_type_ids_list[i]
            pad_length_token = max_length - len(token_type_ids)
            padded_token_type = token_type_ids + [0] * pad_length_token
            padded_token_type_ids.append(padded_token_type)
    
    # 转换为 tensor（Trainer 会自动将数据移动到正确的设备）
    batch_dict = {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(padded_attention_mask, dtype=torch.long),
    }
    
    # 如果有 token_type_ids，也添加到 batch 中
    if padded_token_type_ids:
        batch_dict["token_type_ids"] = torch.tensor(padded_token_type_ids, dtype=torch.long)
    
    # 如果有 labels，也添加到 batch 中
    if labels_list:
        batch_dict["labels"] = torch.tensor(labels_list, dtype=torch.long)
    
    return batch_dict

print("自定义预处理函数已定义")
print("注意：Trainer 会自动将数据移动到模型所在的设备（GPU/CPU）")

自定义预处理函数已定义
注意：Trainer 会自动将数据移动到模型所在的设备（GPU/CPU）


In [26]:
# 对比实验：使用自定义预处理函数训练模型
from transformers import TrainingArguments, Trainer

# 使用自定义的 collate_fn
training_args_custom = TrainingArguments("test-trainer-custom")
model_custom = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model_custom = model_custom.to(device)
print(f"模型已加载到: {next(model_custom.parameters()).device}")

trainer_custom = Trainer(
    model_custom,
    training_args_custom,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=custom_collate_fn,  # 使用自定义的预处理函数
    tokenizer=tokenizer,
)

print("使用自定义预处理函数的 Trainer 已创建")

'(ReadTimeoutError("HTTPSConnectionPool(host='hf-mirror.com', port=443): Read timed out. (read timeout=10)"), '(Request ID: 3d81d46f-3161-4517-87da-ba96676b74cb)')' thrown while requesting HEAD https://hf-mirror.com/bert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


模型已加载到: cuda:0
使用自定义预处理函数的 Trainer 已创建


In [27]:
# 训练并评估（使用自定义预处理）
print("=== 使用自定义预处理函数训练 ===")
print(f"模型设备: {next(trainer_custom.model.parameters()).device}")
trainer_custom.train()

# 评估
print("\n=== 使用自定义预处理函数的评估结果 ===")
eval_results_custom = trainer_custom.evaluate()
print(f"评估结果: {eval_results_custom}")

=== 使用自定义预处理函数训练 ===
模型设备: cuda:0


Step,Training Loss
500,0.615200
1000,0.456700



=== 使用自定义预处理函数的评估结果 ===


评估结果: {'eval_loss': 0.590685248374939, 'eval_runtime': 6.201, 'eval_samples_per_second': 65.796, 'eval_steps_per_second': 8.224, 'epoch': 3.0}


### 对比总结

**使用 DataCollatorWithPadding 的优势：**
- 自动处理 padding，代码更简洁
- 支持动态 padding（只 padding 到批次内最长序列）
- 自动处理 token_type_ids 等字段
- 经过优化，性能更好

**自定义预处理函数：**
- 可以完全控制预处理流程
- 适合特殊需求或学习理解底层实现
- 但需要手动处理各种边界情况

**实际效果对比：**
- 训练效果应该基本相同（都是做 padding）
- 性能上 DataCollatorWithPadding 可能稍快（经过优化）
- 代码可读性上 DataCollatorWithPadding 更好

In [ ]:
# 对比两种方法的结果
print("=" * 60)
print("对比结果总结")
print("=" * 60)

# 如果有之前的评估结果，可以在这里对比
# 使用 DataCollatorWithPadding 的结果（从之前的训练中获取）
print("\n【使用 DataCollatorWithPadding】")
print("优点：代码简洁，自动处理，经过优化")
print("缺点：不够灵活，难以自定义特殊需求")

print("\n【使用自定义预处理函数】")
print("优点：完全可控，可以自定义特殊逻辑")
print("缺点：代码复杂，需要手动处理各种情况")

print("\n【实际效果】")
print("两种方法的训练效果应该基本相同（都是做 padding）")
print("但 DataCollatorWithPadding 在性能和代码可读性上更优")
print("=" * 60)